# NJ-only Candidate Feature List

This notebook documents the candidate features for NJ COVID forecasting. It contains no patient records and does not read the real case workbook. All definitions are embedded here, so teammates can read it without downloading another data file.

**Status:** candidate specification, not an already computed feature table. The final feature subset will be selected using chronological training and validation. NY outcomes and cross-state predictors are excluded from this version. Age and income are excluded.


## Study unit and target

- One final modeling row = one eligible NJ region × one forecast date.
- Current case scaffold: 598 reference identifiers × 1,461 days, covering 2020–2023. Final eligibility still depends on geography and population checks.
- The source field is postal ZIP. Validate the ZIP-to-ZCTA mapping before merging population or commuting features.
- C(i,t) denotes the recorded-onset count after exact-row deduplication. Absent ZIP/date groups are zero under the confirmed source-coverage rule. Records without onset date are excluded from onset aggregation, not assigned to a day.
- Targets are C(i,t+h), h=1,...,14. Future observed counts and trajectories are labels, not predictors.
- About 60.65% of original records lack onset dates. This outcome measures recorded-onset activity; infection-episode semantics and historical availability still need confirmation.


## A. Historical Counts and Calendar

### cases_today

| Item | Description |
|---|---|
| Calculation | C(i,t): NJ ZIP daily recorded-onset count after exact-row deduplication. |
| Meaning | Current local daily case level. |
| Purpose | Short-term persistence. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### cases_lag_1, cases_lag_7, cases_lag_14

| Item | Description |
|---|---|
| Calculation | C(i,t-k), k = 1, 7, 14. |
| Meaning | Cases on specific earlier days, not window totals. |
| Purpose | Local historical predictors. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### day_of_week and month

| Item | Description |
|---|---|
| Calculation | Weekday and month of t; categorical. |
| Meaning | Calendar position of the forecast origin. |
| Purpose | Captures calendar regularity. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### target_weekday and target_month

| Item | Description |
|---|---|
| Calculation | Calendar categories for t+h. |
| Meaning | Known calendar conditions on the predicted day. |
| Purpose | Known-ahead inputs for each horizon. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


## B. Local Case Trends

### cases_sum_7d and cases_sum_14d

| Item | Description |
|---|---|
| Calculation | Sum C(i,t-k), k = 0,...,w-1; w = 7 or 14. |
| Meaning | Recent cumulative new cases. |
| Purpose | Smoother local burden. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### cases_mean_7d and cases_mean_14d

| Item | Description |
|---|---|
| Calculation | Rolling total / w. |
| Meaning | Average daily new cases. |
| Purpose | Optional smoothing alternative. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### case_rate_today_per_100k

| Item | Description |
|---|---|
| Calculation | C(i,t) / N(i) x 100,000. |
| Meaning | Daily cases per 100,000 residents. |
| Purpose | Population-adjusted activity. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### case_rate_7d_per_100k and case_rate_14d_per_100k

| Item | Description |
|---|---|
| Calculation | Rolling case total / N(i) x 100,000. |
| Meaning | Window cumulative rate, not an average daily rate. |
| Purpose | Comparable burden and input to pressure. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### week_over_week_change

| Item | Description |
|---|---|
| Calculation | Sum C(i,t-6:t) minus sum C(i,t-13:t-7). |
| Meaning | Absolute difference between consecutive weekly totals. |
| Purpose | Whether burden is rising or falling. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### case_velocity

| Item | Description |
|---|---|
| Calculation | V(i,t) = C(i,t) - C(i,t-1). |
| Meaning | First-order difference in daily counts. |
| Purpose | Immediate local increase or decrease. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### case_acceleration

| Item | Description |
|---|---|
| Calculation | A(i,t) = V(i,t) - V(i,t-1). |
| Meaning | Second-order difference in daily counts. |
| Purpose | Change in daily growth. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### case_jerk

| Item | Description |
|---|---|
| Calculation | J(i,t) = A(i,t) - A(i,t-1). |
| Meaning | Third-order difference in daily counts. |
| Purpose | Abrupt acceleration changes. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### case_std_7d and case_std_14d

| Item | Description |
|---|---|
| Calculation | Sample standard deviation of daily counts in the full window; ddof = 1. |
| Meaning | Recent case variability. |
| Purpose | Distinguishes stable and irregular periods. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


### days_since_last_observed_case

| Item | Description |
|---|---|
| Calculation | t minus the latest s <= t with a positive recorded-onset count. Zero on positive days; unavailable before the first observed positive day. |
| Meaning | Length of an observed zero-case interval. |
| Purpose | Sparse-series history. |
| Status / requirements | Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date. |


## C. Regional Attributes and NJ Commuting

### population; land_area_sq_km; population_density_per_sq_km

| Item | Description |
|---|---|
| Calculation | ACS population; ALAND / 1,000,000; population / land area. |
| Meaning | NJ resident scale, land area and concentration. |
| Purpose | Rate denominators and regional context. |
| Status / requirements | Existing reference values; require valid ZIP-to-ZCTA mapping. Population/density are static 2021-vintage inputs. No age or income. |


### resident_jobs_nj_network; workplace_jobs_nj_network

| Item | Description |
|---|---|
| Calculation | Row sum and column sum of F(i,j) with BOTH i and j in the eligible NJ reference set, including self-links. |
| Meaning | Jobs represented within the selected NJ network. |
| Purpose | Employment scale in a consistently defined network. |
| Status / requirements | Must recompute; these are not total resident/workplace employment across all destinations/origins. |


### inbound_jobs_external_nj; outbound_jobs_external_nj

| Item | Description |
|---|---|
| Calculation | Column and row sums over eligible NJ regions with i != j. |
| Meaning | Commuting connections between different NJ regions. |
| Purpose | Volume of connections beyond normalized weights. |
| Status / requirements | Must recompute from routes with both endpoints in NJ. External means outside the local region, not outside NJ. |


### internal_jobs

| Item | Description |
|---|---|
| Calculation | F(j,j), residence and work in the same mapped NJ region. |
| Meaning | Within-region jobs. |
| Purpose | Local residence-work overlap. |
| Status / requirements | Reuse after geographic validation; excluded from pressure. |


### external_inbound_share_nj_network; external_outbound_share_nj_network

| Item | Description |
|---|---|
| Calculation | NJ external inflow / workplace_jobs_nj_network; NJ external outflow / resident_jobs_nj_network. |
| Meaning | Dependence on other NJ regions within this network. |
| Purpose | Relative commuting dependence. |
| Status / requirements | Must recompute; zero denominators are unavailable, not zero. Not shares of all actual employment. |


## D. NJ Commuting-Weighted COVID Pressure

### inbound_weight_nj

| Item | Description |
|---|---|
| Calculation | F(i,j) / SUM_k F(k,j), where i and k are eligible NJ origins, j is an eligible NJ destination, and k != j. |
| Meaning | Each NJ origin's share of represented external NJ inflow to j. |
| Purpose | Weight for the NJ-only pressure feature. |
| Status / requirements | Route-level auxiliary field; recompute after restricting both endpoints to NJ. Sums to 1 for destinations with positive external NJ inflow. |


### outbound_weight_nj

| Item | Description |
|---|---|
| Calculation | F(i,j) / SUM_k F(i,k), where k is an eligible NJ destination and k != i. |
| Meaning | Each NJ destination's share of represented external outflow from i. |
| Purpose | Optional network description; not used in inbound pressure. |
| Status / requirements | Route-level auxiliary field; recompute. Not a separate region-day predictor by itself. |


### nj_covid_pressure_7d; nj_covid_pressure_14d

| Item | Description |
|---|---|
| Calculation | SUM_i inbound_weight_nj(i,j) * [SUM_s C(i,t-s) / Population(i) * 100000], s=0,...,w-1, w=7 or 14, i != j. |
| Meaning | NJ commuting-weighted trailing cumulative case rate per 100,000 residents. |
| Purpose | Test whether connected NJ regions improve local forecasts beyond local history. |
| Status / requirements | Planned. NJ-only normalized pressure, different from the old partial NJ contribution under NJ-NY weights. Do not use NY case rates or substitute missing rates with zero. |


### pressure_source_weight_coverage_nj

| Item | Description |
|---|---|
| Calculation | SUM_i inbound_weight_nj(i,j) over origins with a complete w-day case window and valid positive population. |
| Meaning | Share of represented NJ inbound weight with usable case rates. |
| Purpose | Quality diagnostic; distinguish incomplete source data from observed zeros. |
| Status / requirements | Planned diagnostic for each window. Pressure is unavailable when coverage is incomplete unless an explicitly different partial/renormalized measure is defined. No external inflow: weights and pressure unavailable. |


## Important NJ-only pressure rules

Use eligible NJ origins and NJ destinations. Remove within-region self-links from external pressure. Recalculate weights after restricting the network:

$$w^{NJ}_{ij}=\frac{F_{ij}}{\sum_{k\in NJ, k\ne j}F_{kj}},\quad i\ne j$$

$$Pressure^{NJ}_{j,t,w}=\sum_{i\in NJ,i\ne j}w^{NJ}_{ij}\left(\frac{\sum_{s=0}^{w-1}C_{i,t-s}}{N_i}\times100{,}000\right),\quad w\in\{7,14\}$$

These are trailing cumulative case rates, not average daily rates. No NY case rates are used. Old NJ–NY weights cannot simply be filtered and reused as NJ-normalized weights. Destinations without external NJ inflow have undefined weights/pressure. Incomplete source-rate coverage is unavailable, not zero pressure.

When starting with a small NJ study area, select prediction destinations while keeping all eligible NJ origins for pressure unless a different network is explicitly defined. Route weights are intermediate data; the final panel remains region × date.


## Modeling and implementation notes

- Compute all case-history features from trailing information available at the forecast origin. Do not use centered windows or future observations.
- Rolling means are exact rescalings of rolling totals. Avoid including both by default. Velocity is also redundant with today's and yesterday's counts in a linear model.
- Full region fixed effects absorb time-invariant regional attributes. Do not interpret separate static coefficients in that specification.
- Recompute NJ-network job totals and shares. They describe represented NJ connections, not all actual employment including out-of-state connections.
- Population and commuting inputs are static 2021-vintage estimates, not annually updated observations.
- Keep common chronological cutoffs across regions and exclude label windows crossing split boundaries. Fit preprocessing and feature selection on training data, tune using validation data, and reserve testing for final assessment.
- Compare models with and without pressure using comparable targets and splits. PR-AUC and warning lead time evaluate outbreak outputs; residual Moran's I diagnoses geographic structure in forecast errors. MAE/RMSE can supplement daily-count diagnostics.


## Next steps

1. Confirm the counting unit, geographic mapping and eligible NJ cohort.
2. Rebuild the NJ-only commuting network, totals, shares and normalized weights.
3. Join population and generate historical case features and 7/14-day pressure.
4. Create 1–14-day targets and chronological training, validation and testing.
5. Compare Dynamic Panel, XGBoost, Random Forest and any feasible SEIR benchmark.
6. Construct forecast trajectories and outbreak warnings.
7. Test whether COVID-derived regional patterns also appear in other respiratory diseases at compatible dates, geography and units. The current analysis is NJ-to-NJ. NJ-to-NY validation needs adequate NY outcomes.


## Optional: inspect or export the specification in Python

The descriptive cells above are readable without execution. The following cells create a DataFrame from embedded definitions and optionally export a CSV. They do not calculate features or access patient data.


In [ ]:
import pandas as pd

feature_definitions = [{'group': 'A', 'feature': 'cases_today', 'definition': 'C(i,t): NJ ZIP daily recorded-onset count after exact-row deduplication.', 'meaning': 'Current local daily case level.', 'purpose': 'Short-term persistence.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'A', 'feature': 'cases_lag_1, cases_lag_7, cases_lag_14', 'definition': 'C(i,t-k), k = 1, 7, 14.', 'meaning': 'Cases on specific earlier days, not window totals.', 'purpose': 'Local historical predictors.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'A', 'feature': 'day_of_week and month', 'definition': 'Weekday and month of t; categorical.', 'meaning': 'Calendar position of the forecast origin.', 'purpose': 'Captures calendar regularity.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'A', 'feature': 'target_weekday and target_month', 'definition': 'Calendar categories for t+h.', 'meaning': 'Known calendar conditions on the predicted day.', 'purpose': 'Known-ahead inputs for each horizon.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'cases_sum_7d and cases_sum_14d', 'definition': 'Sum C(i,t-k), k = 0,...,w-1; w = 7 or 14.', 'meaning': 'Recent cumulative new cases.', 'purpose': 'Smoother local burden.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'cases_mean_7d and cases_mean_14d', 'definition': 'Rolling total / w.', 'meaning': 'Average daily new cases.', 'purpose': 'Optional smoothing alternative.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'case_rate_today_per_100k', 'definition': 'C(i,t) / N(i) x 100,000.', 'meaning': 'Daily cases per 100,000 residents.', 'purpose': 'Population-adjusted activity.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'case_rate_7d_per_100k and case_rate_14d_per_100k', 'definition': 'Rolling case total / N(i) x 100,000.', 'meaning': 'Window cumulative rate, not an average daily rate.', 'purpose': 'Comparable burden and input to pressure.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'week_over_week_change', 'definition': 'Sum C(i,t-6:t) minus sum C(i,t-13:t-7).', 'meaning': 'Absolute difference between consecutive weekly totals.', 'purpose': 'Whether burden is rising or falling.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'case_velocity', 'definition': 'V(i,t) = C(i,t) - C(i,t-1).', 'meaning': 'First-order difference in daily counts.', 'purpose': 'Immediate local increase or decrease.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'case_acceleration', 'definition': 'A(i,t) = V(i,t) - V(i,t-1).', 'meaning': 'Second-order difference in daily counts.', 'purpose': 'Change in daily growth.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'case_jerk', 'definition': 'J(i,t) = A(i,t) - A(i,t-1).', 'meaning': 'Third-order difference in daily counts.', 'purpose': 'Abrupt acceleration changes.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'case_std_7d and case_std_14d', 'definition': 'Sample standard deviation of daily counts in the full window; ddof = 1.', 'meaning': 'Recent case variability.', 'purpose': 'Distinguishes stable and irregular periods.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'B', 'feature': 'days_since_last_observed_case', 'definition': 't minus the latest s <= t with a positive recorded-onset count. Zero on positive days; unavailable before the first observed positive day.', 'meaning': 'Length of an observed zero-case interval.', 'purpose': 'Sparse-series history.', 'availability': 'Planned: calculate from the real NJ balanced onset panel after the counting unit is finalized. Use only history available by the forecast date.'}, {'group': 'C', 'feature': 'population; land_area_sq_km; population_density_per_sq_km', 'definition': 'ACS population; ALAND / 1,000,000; population / land area.', 'meaning': 'NJ resident scale, land area and concentration.', 'purpose': 'Rate denominators and regional context.', 'availability': 'Existing reference values; require valid ZIP-to-ZCTA mapping. Population/density are static 2021-vintage inputs. No age or income.'}, {'group': 'C', 'feature': 'resident_jobs_nj_network; workplace_jobs_nj_network', 'definition': 'Row sum and column sum of F(i,j) with BOTH i and j in the eligible NJ reference set, including self-links.', 'meaning': 'Jobs represented within the selected NJ network.', 'purpose': 'Employment scale in a consistently defined network.', 'availability': 'Must recompute; these are not total resident/workplace employment across all destinations/origins.'}, {'group': 'C', 'feature': 'inbound_jobs_external_nj; outbound_jobs_external_nj', 'definition': 'Column and row sums over eligible NJ regions with i != j.', 'meaning': 'Commuting connections between different NJ regions.', 'purpose': 'Volume of connections beyond normalized weights.', 'availability': 'Must recompute from routes with both endpoints in NJ. External means outside the local region, not outside NJ.'}, {'group': 'C', 'feature': 'internal_jobs', 'definition': 'F(j,j), residence and work in the same mapped NJ region.', 'meaning': 'Within-region jobs.', 'purpose': 'Local residence-work overlap.', 'availability': 'Reuse after geographic validation; excluded from pressure.'}, {'group': 'C', 'feature': 'external_inbound_share_nj_network; external_outbound_share_nj_network', 'definition': 'NJ external inflow / workplace_jobs_nj_network; NJ external outflow / resident_jobs_nj_network.', 'meaning': 'Dependence on other NJ regions within this network.', 'purpose': 'Relative commuting dependence.', 'availability': 'Must recompute; zero denominators are unavailable, not zero. Not shares of all actual employment.'}, {'group': 'D', 'feature': 'inbound_weight_nj', 'definition': 'F(i,j) / SUM_k F(k,j), where i and k are eligible NJ origins, j is an eligible NJ destination, and k != j.', 'meaning': "Each NJ origin's share of represented external NJ inflow to j.", 'purpose': 'Weight for the NJ-only pressure feature.', 'availability': 'Route-level auxiliary field; recompute after restricting both endpoints to NJ. Sums to 1 for destinations with positive external NJ inflow.'}, {'group': 'D', 'feature': 'outbound_weight_nj', 'definition': 'F(i,j) / SUM_k F(i,k), where k is an eligible NJ destination and k != i.', 'meaning': "Each NJ destination's share of represented external outflow from i.", 'purpose': 'Optional network description; not used in inbound pressure.', 'availability': 'Route-level auxiliary field; recompute. Not a separate region-day predictor by itself.'}, {'group': 'D', 'feature': 'nj_covid_pressure_7d; nj_covid_pressure_14d', 'definition': 'SUM_i inbound_weight_nj(i,j) * [SUM_s C(i,t-s) / Population(i) * 100000], s=0,...,w-1, w=7 or 14, i != j.', 'meaning': 'NJ commuting-weighted trailing cumulative case rate per 100,000 residents.', 'purpose': 'Test whether connected NJ regions improve local forecasts beyond local history.', 'availability': 'Planned. NJ-only normalized pressure, different from the old partial NJ contribution under NJ-NY weights. Do not use NY case rates or substitute missing rates with zero.'}, {'group': 'D', 'feature': 'pressure_source_weight_coverage_nj', 'definition': 'SUM_i inbound_weight_nj(i,j) over origins with a complete w-day case window and valid positive population.', 'meaning': 'Share of represented NJ inbound weight with usable case rates.', 'purpose': 'Quality diagnostic; distinguish incomplete source data from observed zeros.', 'availability': 'Planned diagnostic for each window. Pressure is unavailable when coverage is incomplete unless an explicitly different partial/renormalized measure is defined. No external inflow: weights and pressure unavailable.'}]

feature_list = pd.DataFrame(feature_definitions)
display(feature_list)


In [ ]:
# Optional export beside this notebook. Uncomment to save.
# from pathlib import Path
# feature_list.to_csv(Path.cwd() / "NJ_only_candidate_feature_list.csv", index=False, encoding="utf-8-sig")
